## 1. Environment & Path Setup
This cell dynamically checks if you are running in the cloud natively (Google Colab) or locally (Antigravity IDE), and sets the root working directory accordingly without crashing.

In [ ]:
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("Running in Google Colab. Mounting Google Drive...")
    from google.colab import drive
    drive.mount('/content/drive')
    # Google Drive structure assumption
    PROJECT_PATH = '/content/drive/MyDrive/androidrec/Android_Recommender_System'
    if os.path.exists(PROJECT_PATH):
        os.chdir(PROJECT_PATH)
        print(f"\nSuccessfully targeted project directory: {os.getcwd()}")
    else:
        print(f"\n[ERROR] Could not find {PROJECT_PATH} in Google Drive.")
        print("Please upload the Android_Recommender_System folder directly into your root MyDrive.")
else:
    print("Running locally (e.g. Antigravity IDE). Retaining current project root:")
    print(os.getcwd())

## 2. Dependency Installation & GPU Verification
Google Colab already provisions `torch`, `numpy`, and `scikit-learn`. Forcing manual re-installs of Torch pip wheels can destroy cloud GPU bindings. This cell only installs what is strictly missing (`torch-geometric`) and performs safety checks.

In [ ]:
import torch

print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Name:", torch.cuda.get_device_name(0))
else:
    print("\n[CRITICAL WARNING] No GPU detected! Your graph neural network will train extremely slowly.")
    print("If on Google Colab: Go to 'Runtime' > 'Change runtime type' > Select 'T4 GPU'\n")

if IN_COLAB:
    # Quiet installation of remaining libraries required by the Colab cloud
    !pip install -q torch-geometric scikit-surprise joblib


## 3. Data Pipeline Intercept Check
The graph requires `.npz` binary matrices to execute. If they haven't been constructed yet (e.g. fresh VM instance), this safely auto-triggers the build script.

In [ ]:
!python variant_graph/graph_build_npy.py


## 4. Execute LightGCN on GPU
Fires the CUDA-optimized graph variant script directly locally. Metrics will output mathematically identically.

In [ ]:
!python variant_graph/variant_lightgcn_colab.py

In [ ]:
!python variant_graph/variant_lightgcn_metadata_colab.py